# Fongbe Challenge — Setup audio (extraction Drive → disque local Colab)

1. Monte Drive et copie les CSV/scripts sur le disque local.
2. Extrait les archives .tar **sélectionnées** vers `/content/fongbe/audio/` (disque local = rapide pour l'entraînement).
3. Vérifie que les WAV correspondent aux CSV.

Relançable : les WAV déjà extraits sont ignorés.

In [ ]:
from pathlib import Path
from google.colab import drive
import shutil, subprocess, sys

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/fongbe-challenge')
TAR_DIR    = DRIVE_ROOT / 'audio'
DATA_DIR   = Path('/content/fongbe')          # CSV + scripts + audio/, sur disque local
AUDIO_DIR  = DATA_DIR                          # les .tar contiennent déjà 'audio/...'

# --- Ce qu'on extrait cette fois (modifiable) ---
# Groupes dispo : 'test', 'reliable', 'religious', 'uncertain'
GROUPS = ['reliable']        # ~13 GB de WAV — suffisant pour le 1er fine-tuning
# ----------------------------------------------

DATA_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# Copie les petits fichiers (CSV, scripts) depuis Drive → local
for f in (DRIVE_ROOT / 'data').iterdir():
    dst = DATA_DIR / f.name
    if f.suffix in ('.csv', '.py', '.md') and not dst.exists():
        shutil.copy2(f, dst)

print(f'✓ DATA_DIR = {DATA_DIR}')
print(f'✓ {len(list(DATA_DIR.glob("*.csv")))} CSV copiés')

In [ ]:
# Sélection des archives selon GROUPS
GROUP_MAP = {
    'test':       ['audio_test_'],
    'reliable':   ['audio_train_reliable_'],
    'religious':  ['audio_train_religious_'],
    'uncertain':  ['audio_train_uncertain_'],
}
prefixes = [p for g in GROUPS for p in GROUP_MAP[g]]
tars = sorted(p for p in TAR_DIR.glob('*.tar') if any(p.name.startswith(x) for x in prefixes))

# Estimation de l'espace disque local nécessaire
import shutil as _sh
free_gb = _sh.disk_usage('/content').free / 1e9
tar_gb = sum(p.stat().st_size for p in tars) / 1e9
print(f'{len(tars)} archives (~{tar_gb:.1f} GB tar → WAV similaire)')
print(f'Espace disque local libre : {free_gb:.0f} GB')
assert tar_gb < free_gb * 0.9, 'Disque local insuffisant : réduis GROUPS'
tars[:3], '...', tars[-1:]

In [ ]:
# Extraction (reprise possible : on saute les .tar déjà extraits via un marqueur)
DONE_DIR = AUDIO_DIR / '.extracted'
DONE_DIR.mkdir(exist_ok=True)

for tar in tars:
    marker = DONE_DIR / tar.name
    if marker.exists():
        print(f'↷ skip {tar.name}')
        continue
    print(f'⬇️  extraction {tar.name} ...', flush=True)
    # lecture directe depuis Drive, écriture sur disque local
    subprocess.run(['tar', '-xf', str(tar), '-C', str(AUDIO_DIR)], check=True)
    marker.touch()
    print(f'✓ {tar.name}')

print('✅ Extraction terminée')

In [ ]:
# Vérification : chaque audio_path du CSV existe sur le disque local
import pandas as pd

checks = {
    'reliable':   'Train_reliable.csv',
    'religious':  'Train_religious.csv',
    'uncertain':  'Train_uncertain.csv',
    'test':       'Test.csv',
}
folder_of = {'reliable': 'audio/train_reliable', 'religious': 'audio/train_religious',
             'uncertain': 'audio/train_uncertain', 'test': 'audio/test'}

for group, csv_name in checks.items():
    sub = AUDIO_DIR / folder_of[group]
    n_wav = len(list(sub.glob('*.wav'))) if sub.exists() else 0
    extracted = group in GROUPS
    status = f'{n_wav:,} wav' if extracted else '(non extrait)'
    print(f'{group:10s} {status}')

# Détail pour les groupes extraits
for group in GROUPS:
    df = pd.read_csv(DATA_DIR / checks[group])
    missing = (~df.audio_path.map(lambda p: (AUDIO_DIR / p).exists())).sum()
    print(f'{group}: {len(df):,} lignes CSV, {missing} wav manquants')

## Suite

Setup OK → ouvrir `colab_train_w2vbert.ipynb` pour le fine-tuning baseline.
Pour ajouter un groupe plus tard : modifier `GROUPS` et relancer ce notebook.